# Whale Calls — ResNet18 Training (Colab)
Before running: set **Runtime → Change runtime type → GPU**.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — go to Runtime → Change runtime type → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone https://github.com/shaishechter/whale_calls.git /content/whale_calls
%cd /content/whale_calls
!pip install -r requirements.txt -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this to the path of your dataset folder inside Google Drive
DATA_PATH = "/content/drive/MyDrive/deep_voice"

In [ ]:
import wandb
wandb.login()

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()  # safe to re-run this cell

with initialize_config_dir(config_dir="/content/whale_calls/conf", version_base=None):
    cfg = compose(
        config_name="resnet18",
        overrides=[f"datamodule.base_path={DATA_PATH}"],
    )

print(OmegaConf.to_yaml(cfg))

In [ ]:
from hydra.utils import instantiate

datamodule = instantiate(cfg.datamodule)
datamodule.setup()

model = instantiate(cfg.model)

logger = instantiate(cfg.trainer.logger)
logger.log_hyperparams(model.hparams)

trainer = instantiate(cfg.trainer, logger=logger)
trainer.fit(model, datamodule=datamodule)